# 02 — Benchmark Reproduction (SST-2)

This notebook reproduces the headline SST-2 benchmark from the README. It runs two pipelines back-to-back on the same data, with the same seed, and produces a side-by-side comparison.

**Configs used:**
- `configs/benchmark_sst2_baseline.yaml` — Wake-only fine-tuning (matched epoch budget)
- `configs/benchmark_sst2.yaml` — Full Wake -> Dream -> Nightmare -> Compress cycle

**Runtime:** ~15-25 minutes on CPU, ~3-5 minutes on a T4 GPU. Reduce `dataset.max_samples` in the configs for faster iteration.

**GPU:** Strongly recommended. CPU works but is slow. On Colab, choose Runtime -> Change runtime type -> T4 GPU.

**What you will produce:**
1. Two trained checkpoints (baseline and 4-phase) with deterministic seeds
2. A markdown table identical in shape to the one in `README.md`
3. A bar chart comparing clean vs robustness scores

## 1. Install and locate the repo

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_ROOT = None
for candidate in ['..', '.', '/content/NightmareNet']:
    if os.path.exists(os.path.join(candidate, 'pyproject.toml')):
        REPO_ROOT = Path(candidate).resolve()
        break

if REPO_ROOT is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nightmarenet'])
    raise RuntimeError(
        'Repo not found. Clone first:\n'
        '  !git clone https://github.com/Adit-Jain-srm/NightmareNet.git\n'
        '  %cd NightmareNet'
    )

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_ROOT)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'datasets', 'matplotlib'])

print(f'Repo root: {REPO_ROOT}')

## 2. Load the two configs

Both configs slice SST-2 to 2,000 samples by default. We preserve that for an apples-to-apples comparison. If you want a quick smoke test, override `max_samples` to 200.

In [ ]:
import torch
from nightmarenet.utils.config import load_config

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

baseline_path = REPO_ROOT / 'configs' / 'benchmark_sst2_baseline.yaml'
fullcycle_path = REPO_ROOT / 'configs' / 'benchmark_sst2.yaml'

baseline_cfg = load_config(str(baseline_path))
fullcycle_cfg = load_config(str(fullcycle_path))

for cfg in (baseline_cfg, fullcycle_cfg):
    cfg['training']['device'] = device

FAST_MODE = False
if FAST_MODE:
    for cfg in (baseline_cfg, fullcycle_cfg):
        cfg['dataset']['max_samples'] = 200
        cfg['training']['wake_epochs'] = max(1, cfg['training'].get('wake_epochs', 1) // 2)

print('Baseline training plan:', baseline_cfg['training'])
print('Full-cycle training plan:', fullcycle_cfg['training'])

## 3. Run the baseline pipeline (Wake only)

In [ ]:
from nightmarenet.pipeline import Pipeline

def run_pipeline(config, label):
    print(f'\n=== Running {label} ===')
    history = []

    def on_event(event):
        phase = event.get('current_phase', '')
        loss = event.get('phase_loss')
        if phase and loss is not None:
            history.append({'phase': phase, 'loss': float(loss)})

    pipe = Pipeline(config=config, on_event=on_event)
    pipe.ingest(
        hf_dataset=config['dataset'].get('name', 'glue'),
        hf_subset=config['dataset'].get('config'),
    )
    pipe.prepare()
    pipe.train()
    comparison = pipe.evaluate()
    return pipe, comparison, history

baseline_pipe, baseline_cmp, baseline_history = run_pipeline(baseline_cfg, 'BASELINE')

## 4. Run the full 4-phase cycle

In [ ]:
fullcycle_pipe, fullcycle_cmp, fullcycle_history = run_pipeline(fullcycle_cfg, 'NIGHTMARENET (4-phase)')

## 5. Compare the two

We extract the trained-model metrics from each pipeline and assemble a table. The exact metric keys depend on the evaluator, so we coalesce the most common ones (clean accuracy, robustness score) and fall back to whatever is present.

In [ ]:
def extract_metrics(pipe):
    trained = pipe.metrics.trained_results or {}
    out = {}
    recall = trained.get('recall') if isinstance(trained.get('recall'), dict) else {}
    robustness = trained.get('robustness') if isinstance(trained.get('robustness'), dict) else {}
    out['clean_acc'] = recall.get('accuracy', recall.get('recall'))
    out['robustness'] = robustness.get('score', robustness.get('robustness_score'))
    return out

rows = [
    ('Baseline (Wake only)', extract_metrics(baseline_pipe)),
    ('NightmareNet (4-phase)', extract_metrics(fullcycle_pipe)),
]

header = '| Method | Clean Acc | Robustness Score |'
sep = '|--------|-----------|------------------|'
lines = [header, sep]
for label, metrics in rows:
    clean = metrics.get('clean_acc')
    rob = metrics.get('robustness')
    clean_s = f'{clean:.3f}' if isinstance(clean, (int, float)) else 'n/a'
    rob_s = f'{rob:.3f}' if isinstance(rob, (int, float)) else 'n/a'
    lines.append(f'| {label} | {clean_s} | {rob_s} |')

table_md = '\n'.join(lines)
print(table_md)

## 6. Visualise the delta

In [ ]:
import matplotlib.pyplot as plt

labels = [row[0] for row in rows]
clean = [row[1].get('clean_acc') or 0.0 for row in rows]
rob = [row[1].get('robustness') or 0.0 for row in rows]

x = range(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([i - width / 2 for i in x], clean, width, label='Clean accuracy', color='#06B6D4')
ax.bar([i + width / 2 for i in x], rob, width, label='Robustness score', color='#EF4444')
ax.set_xticks(list(x))
ax.set_xticklabels(labels, rotation=10, ha='right')
ax.set_ylim(0, 1.0)
ax.set_ylabel('Score')
ax.set_title('SST-2: Baseline vs NightmareNet 4-phase cycle')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Save the comparison

Write the table to `results/benchmark-sst2-reproduction.md` for your own records. This is the same shape the README pulls from.

In [ ]:
out_dir = Path('results')
out_dir.mkdir(exist_ok=True)
out_path = out_dir / 'benchmark-sst2-reproduction.md'
out_path.write_text(
    '# SST-2 Benchmark Reproduction\n\n'
    f'Device: `{device}` | Dataset slice: `{baseline_cfg["dataset"].get("max_samples")}`\n\n'
    + table_md + '\n',
    encoding='utf-8',
)
print(f'Saved -> {out_path.resolve()}')

## Next steps

- **`03_custom_distortions.ipynb`** — write your own distortion plugin and evaluate a model with it. This is the recommended next step for researchers extending the framework.
- For larger sweeps, look at `configs/benchmark_sst2_gpu.yaml` and `configs/benchmark_sst2_baseline_gpu.yaml`.
- The full benchmark methodology lives in [`docs/research/benchmark-v1.md`](../docs/research/benchmark-v1.md).
- For paper-quality results, run multiple seeds (42, 123, 7) and report mean +/- std.